# 👾 ElatoAI: Running Grok Realtime Speech on ESP32 (Arduino) with Deno Edge Functions

This guide shows how to build an AI voice agent device with realtime speech powered by **Grok Realtime API**, ESP32, secure WebSockets, and Deno edge functions for long-running global conversations.

Active project repository: [ElatoAI](https://github.com/akdeb/ElatoAI)

[![Elato AI Demo Video](https://raw.githubusercontent.com/akdeb/ElatoAI/refs/heads/main/assets/thumbnail.png)](https://www.youtube.com/watch?v=o1eIAwVll5I)



## ⚡️ DIY Hardware Design

The reference implementation uses an ESP32-S3 microcontroller with minimal additional components.

**Required Components:**
- ESP32-S3 development board
- I2S microphone (e.g., INMP441)
- I2S amplifier and speaker (e.g., MAX98357A)
- Push button to start/stop the conversation
- RGB LED for visual feedback
- Optional: touch sensor for alternative control

**Hardware options:**
A fully assembled PCB and device are available in the [ElatoAI store](https://www.elatoai.com/products).



## 📱 App Design

Control your ESP32 AI device from your phone with your own web app.

Core app capabilities:
- Select from a list of AI characters
- Talk to your AI with realtime responses
- Create personalized AI characters
- Manage linked devices and settings



## ✨ Quick Start Tutorial

Video tutorial: [Watch on YouTube](https://www.youtube.com/watch?v=bXrNRpGOJWw)

### 1. Clone the repository

```bash
git clone https://github.com/akdeb/ElatoAI.git
cd ElatoAI
```

### 2. Set environment variables (`XAI_API_KEY`, `SUPABASE_ANON_KEY`, `SUPABASE_KEY`)

In `frontend-nextjs`:

```bash
cd frontend-nextjs
cp .env.example .env.local

# .env.local
# NEXT_PUBLIC_SUPABASE_ANON_KEY=<your-supabase-anon-key>
# XAI_API_KEY=<your-xai-api-key>
```

In `server-deno`:

```bash
cd server-deno
cp .env.example .env

# .env
# SUPABASE_KEY=<your-supabase-service-key>
# XAI_API_KEY=<your-xai-api-key>
```

### 3. Start Supabase

Install Supabase CLI and run your local backend:

```bash
brew install supabase/tap/supabase
supabase start
```

### 4. Set up the Next.js frontend

Frontend docs: [frontend-nextjs/README.md](https://github.com/akdeb/ElatoAI/tree/main/frontend-nextjs/README.md)

```bash
cd frontend-nextjs
npm install
npm run dev
```

### 5. Start the Deno server

Server docs: [server-deno/README.md](https://github.com/akdeb/ElatoAI/tree/main/server-deno/README.md)

```bash
cd server-deno
deno run -A --env-file=.env main.ts
```

### 6. Set up ESP32 firmware

Firmware docs: [firmware-arduino/README.md](https://github.com/akdeb/ElatoAI/tree/main/firmware-arduino/README.md)

In `Config.cpp`, set `ws_server` and `backend_server` to your local IP address.

```bash
ifconfig
```

Find your Wi-Fi interface IP (for example `192.168.1.100`) and use it for local service connection.

### 7. Configure ESP32 Wi-Fi

Build and upload firmware. The device should expose an `ELATO-DEVICE` captive portal.
Open `http://192.168.4.1` and configure Wi-Fi credentials.

### 8. Reboot the device and test

After Wi-Fi setup, power-cycle the device. It should reconnect and be ready for conversations.



## 🚀 Ready to Launch?

1. Register your device by adding the ESP32 MAC address and a unique user code in the `devices` table in Supabase.
2. In frontend settings, add the same user code to link the device to your account.
3. For local testing, keep `DEV_MODE` enabled in `firmware-arduino/Config.h` and use local IPs in server config.
4. Repeat the process to register additional devices.

**Pro tip:** To read the ESP32-S3 MAC address, upload `test/print_mac_address_test.cpp` and inspect serial monitor output.



## Project Architecture

ElatoAI consists of three main components:

1. **Frontend Client** (`Next.js` on Vercel): create/manage AI characters and send sessions to ESP32
2. **Edge Server Functions** (`Deno` on Deno/Supabase Edge): websocket handling + Grok API calls
3. **ESP32 IoT Client** (`PlatformIO/Arduino`): streams audio to/from edge server



## Grok Realtime Implementation Details

### Realtime endpoint
- `wss://api.x.ai/v1/realtime`

### Session setup (server VAD)

```ts
const XAI_REALTIME_URL = "wss://api.x.ai/v1/realtime";

const grokWs = new WebSocket(XAI_REALTIME_URL, {
  headers: {
    Authorization: `Bearer ${xaiApiKey}`,
    "Content-Type": "application/json",
  },
});

grokWs.send(
  JSON.stringify({
    type: "session.update",
    session: {
      voice,
      instructions: systemPrompt,
      turn_detection: { type: "server_vad" },
      audio: {
        input: { format: { type: "audio/pcm", rate: 16000 } },
        output: { format: { type: "audio/pcm", rate: 24000 } },
      },
    },
  }),
);
```

### Event handling model

The Grok relay logic uses:
- `input_audio_buffer.append` for streaming microphone chunks
- `input_audio_buffer.speech_started` / `speech_stopped` for VAD lifecycle
- `response.created` to notify device playback start (`RESPONSE.CREATED`)
- `response.output_audio.delta` to decode base64 PCM and packetize as Opus for ESP32
- `response.output_audio_transcript.delta` to accumulate assistant transcript
- `conversation.item.input_audio_transcription.completed` to persist user transcript
- `response.done` to flush final audio, save assistant transcript, and send `RESPONSE.COMPLETE`
- `error` to send `RESPONSE.ERROR`

### ESP32 bridge behavior

- Binary frames from ESP32 are forwarded as Grok input audio
- Instruction messages are parsed from JSON control frames
- `INTERRUPT` clears Grok input buffer (`input_audio_buffer.clear`)
- Manual end-of-speech commit is **not** required when `server_vad` is enabled



## 🌟 Key Features

1. **Realtime Speech-to-Speech** powered by Grok Realtime API
2. **Custom AI Agents** with configurable personalities and voices
3. **Secure WebSockets** for encrypted streaming
4. **Server VAD Turn Detection** for smooth turn-taking
5. **Opus Audio Compression** for efficient bandwidth usage
6. **Global Edge Performance** using Deno edge runtime
7. **ESP32 Arduino Framework** optimized for embedded deployment
8. **Conversation History** persisted in Supabase
9. **Device Management + Authentication** through app + DB workflows
10. **WebRTC and WebSocket Conversation Paths** (web + ESP32)
11. **Volume Control** from Next.js app
12. **Realtime Transcripts** saved in Supabase
13. **OTA Firmware Updates**
14. **Wi-Fi Captive Portal Setup**
15. **Factory Reset Support**
16. **Button and Touch Input Support**
17. **No PSRAM Required** for supported speech-to-speech path
18. **OAuth for Web Client**



## 🛠 Tech Stack

| Component       | Technology Used |
|-----------------|-----------------|
| Frontend        | Next.js, Vercel |
| Backend         | Supabase DB |
| Edge Functions  | Deno / Supabase Edge Runtime |
| IoT Client      | PlatformIO, Arduino Framework, ESP32-S3 |
| Audio Codec     | Opus |
| Communication   | Secure WebSockets |
| Libraries       | ArduinoJson, WebSockets, AsyncWebServer, ESP32_Button, Arduino Audio Tools, ArduinoLibOpus |



## ⚙️ PlatformIO Config

```ini
[env:esp32-s3-devkitc-1]
platform = espressif32 @ 6.10.0
board = esp32-s3-devkitc-1
framework = arduino
monitor_speed = 115200

lib_deps =
    bblanchon/ArduinoJson@^7.1.0
    links2004/WebSockets@^2.4.1
    ESP32Async/ESPAsyncWebServer@^3.7.6
    https://github.com/esp-arduino-libs/ESP32_Button.git#v0.0.1
    https://github.com/pschatzmann/arduino-audio-tools.git#v1.0.1
    https://github.com/pschatzmann/arduino-libopus.git#a1.1.0
```



## 📊 Important Stats

- ⚡️ **Latency**: typically under ~2s round-trip (network-dependent)
- 🎧 **Audio Quality**: Opus codec at ~12kbps target profile
- ⏳ **Continuous Session Window**: long-running sessions with edge runtime limits
- 🌎 **Global Availability**: edge-distributed architecture



## 🛡 Security

- Secure WebSockets (WSS) for encrypted transport
- Optional API key encryption (AES-256) at rest/in transit in your own infrastructure
- Supabase auth + RLS-protected table access
- Device registration controls via unique code + MAC mapping



## 🚫 Limitations

- Cold start delays can occur when edge functions spin up
- Session duration is bounded by edge runtime wall-clock limits
- Speech interruption behavior depends on current ESP32/client signaling path
- End-to-end quality is sensitive to Wi-Fi quality and packet loss



## 📈 Core Use Cases

Use-case catalog: [Usecases.md](https://github.com/akdeb/ElatoAI/tree/main/Usecases.md)

Typical scenarios:
- AI companion toys
- Education assistants
- Smart home voice surfaces
- Retail and kiosk conversational endpoints
- Embedded multilingual assistants



## License

This project is licensed under the MIT License. See upstream [LICENSE](https://github.com/akdeb/ElatoAI/blob/main/LICENSE).

---

For full project details and latest updates, visit [ElatoAI](https://github.com/akdeb/ElatoAI).

